##1. Setup e Configuração
Importação de bibliotecas essenciais e definição dinâmica do caminho dos dados no Repositório.

In [0]:
import os
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType, IntegerType

current_path = os.getcwd()
repo_name = "Grupo7-Setor-de-Seguros"

if repo_name in current_path:
    root_path = current_path.split(repo_name)[0] + repo_name
else:
    root_path = os.path.dirname(os.path.dirname(os.getcwd()))

caminho_arquivo = f"{root_path}/data/processed/prata/seguros_sinistros.csv"

print(f"📂 Diretório Raiz: {root_path}")
print(f"🎯 Arquivo Alvo: {caminho_arquivo}")

## 2. Leitura e Sanitização (Data Cleaning)
Leitura via Pandas (para contornar bloqueio de segurança) e conversão para Spark utilizando `try_cast`.
* **Objetivo:** Garantir que erros de formatação no CSV não quebrem a análise. Textos em colunas numéricas serão convertidos para `0.0`.

In [0]:
print("--- INICIANDO LEITURA BLINDADA ---")

try:
    pdf = pd.read_csv(caminho_arquivo, sep=',', decimal='.', dtype=str, on_bad_lines='skip')
    
    pdf = pdf.where(pd.notnull(pdf), None)
    
    df_full_raw = spark.createDataFrame(pdf)
    
    cols_numericas = ["valor_premio", "valor_pagamento", "capital_segurado", "valor_sinistro"]
    
    df_full = df_full_raw
    for col_name in cols_numericas:
        if col_name in df_full.columns:
            df_full = df_full.withColumn(
                col_name, 
                F.coalesce(F.expr(f"try_cast({col_name} as DOUBLE)"), F.lit(0.0))
            )

    for col_text in ["nome_contratante", "nome_beneficiario", "tipo_sinistro"]:
        if col_text in df_full.columns:
            df_full = df_full.withColumn(col_text, F.trim(F.col(col_text)))

    df_sinistros = df_full.filter(F.col("tipo_sinistro").isNotNull()) \
    .dropDuplicates(["nome_contratante", "data_contratacao", "valor_sinistro", "tipo_sinistro"])
    df_seguros = df_full.dropDuplicates(["nome_contratante", "valor_premio", "capital_segurado"])

    print(f"✅ Dados Carregados com Sucesso!")
    print(f"📊 Contratos Únicos: {df_seguros.count()}")
    print(f"💥 Sinistros Registrados: {df_sinistros.count()}")
    
except Exception as e:
    print(f"❌ Erro Crítico na Leitura: {e}")

## 3. Engenharia de Atributos
* **REGIAO:** Agrupamento de estados.
* **SEXO:** Classificação via Primeiro Nome (IBGE + Heurística).
* **QUARTIS:** Segmentação de Capital.

In [0]:
# Seus imports atuais
import pyspark.sql.functions as F

# 1. Região
df_seguros_prep = df_seguros.withColumn("REGIAO", 
    F.when(F.col("estado_contratante").isin("SP", "RJ", "MG", "ES"), "Sudeste")
    .when(F.col("estado_contratante").isin("PR", "SC", "RS"), "Sul")
    .when(F.col("estado_contratante").isin("PE", "BA", "CE", "MA", "PB", "AL", "SE", "RN", "PI"), "Nordeste")
    .when(F.col("estado_contratante").isin("AM", "PA", "AC", "RR", "RO", "TO", "AP"), "Norte")
    .otherwise("Centro-Oeste")
)

# 2. Sexo (Correção Primeiro Nome)
df_seguros_prep = df_seguros_prep.withColumn("primeiro_nome", F.split(F.col("nome_contratante"), " ").getItem(0))

nomes_fem_excecao = ["Julie", "Alice", "Beatriz", "Ines", "Raquel", "Liz", "Isabel"]
df_seguros_prep = df_seguros_prep.withColumn("SEXO",
    F.when(F.col("primeiro_nome").isin(nomes_fem_excecao), "Feminino")
    .when(F.lower(F.col("primeiro_nome")).endswith("a"), "Feminino")
    .otherwise("Masculino")
).drop("primeiro_nome")

# 3. Quartis
quartis = df_seguros_prep.approxQuantile("capital_segurado", [0.25, 0.75], 0.01)
df_seguros_prep = df_seguros_prep \
    .withColumn("ACIMA_Q3_CAPITAL", F.col("capital_segurado") > F.lit(quartis[1])) \
    .withColumn("ABAIXO_Q1_CAPITAL", F.col("capital_segurado") < F.lit(quartis[0]))

print("✅ Variável 'df_seguros_prep' criada na memória!")
display(df_seguros_prep.limit(2))

## Task A: Análise de Autosseguro
Verificação de casos onde Contratante e Beneficiário são a mesma pessoa.
> **Nota:** Se o resultado for 0, indica uma regra de negócio da base onde os papéis são excludentes por contrato.

In [0]:
import pyspark.sql.functions as F

print("--- TASK A ---")

df_auto = df_prep.filter(F.upper(F.col("nome_contratante")) == F.upper(F.col("nome_beneficiario")))
count_auto = df_auto.count()
print(f"Registros encontrados: {count_auto}")

if count_auto > 0:
    print("\n1. Correlações por Segmento (Trimestre/Sexo):")
    
    segmentos = df_auto.select("TRIMESTRE", "SEXO").distinct().collect()
    
    for row in segmentos:
        t, s = row['TRIMESTRE'], row['SEXO']
        df_seg = df_auto.filter((F.col("TRIMESTRE")==t) & (F.col("SEXO")==s))
        
        if df_seg.count() > 1:
            try:
                corr_premio_cap = df_seg.stat.corr("valor_premio", "capital_segurado")
                corr_pag_premio = df_seg.stat.corr("valor_pagamento", "valor_premio")
                print(f"[{s} | Trimestre {t}] Corr(Prêmio x Capital): {corr_premio_cap:.4f} | Corr(Pag x Prêmio): {corr_pag_premio:.4f}")
            except Exception as e:
                print(f"[{s} | Trimestre {t}] Não foi possível calcular correlação (Dados insuficientes ou constantes).")

    print("\n2. Estatísticas da Razão (Prêmio / Pagamento):")
    
    df_stats = df_auto.withColumn("razao_premio_pgt", F.col("valor_premio") / F.col("valor_pagamento"))
    
    df_stats.select("razao_premio_pgt").summary("mean", "50%", "stddev", "min", "max").show()

    print("\n3. Frequência por Região:")
    display(df_auto.groupBy("REGIAO").count().orderBy(F.col("count").desc()))
    
else:
    print("⚠️ Nenhum caso de autosseguro encontrado (Contratante == Beneficiário).")
    print("As sub-tarefas de correlação e estatística não puderam ser calculadas pois o conjunto de dados é vazio.")

## Task B: Clientes Mais Frequentes (Heavy Users)
Identificação do perfil de risco (Top 10% em frequência de acidentes).

In [0]:
df_freq = df_sinistros.groupBy("nome_contratante").agg(F.count("tipo_sinistro").alias("qtd_acidentes"))

corte_alto = max(df_freq.approxQuantile("qtd_acidentes", [0.90], 0.01)[0], 2.0)
corte_baixo = df_freq.approxQuantile("qtd_acidentes", [0.10], 0.01)[0]

df_freq = df_freq.withColumn("TIPO_CLIENTE", 
    F.when(F.col("qtd_acidentes") >= corte_alto, "Mais Frequente")
    .when(F.col("qtd_acidentes") <= corte_baixo, "Menos Frequente")
    .otherwise("Médio")
)

df_perfil_B = df_freq.join(df_prep, "nome_contratante", "inner") \
    .withColumn("razao_pgt_capital", F.col("valor_pagamento") / F.col("capital_segurado"))

cols_show = ["nome_contratante", "TIPO_CLIENTE", "REGIAO", "SEXO", "qtd_acidentes", "razao_pgt_capital"]

print(f"--- Clientes Mais Frequentes (>= {corte_alto} acidentes) ---")
display(df_perfil_B.filter(F.col("TIPO_CLIENTE") == "Mais Frequente")
        .select(cols_show).dropDuplicates(["nome_contratante"]).limit(5))

print(f"--- Clientes Menos Frequentes (<= {corte_baixo} acidentes) ---")
display(df_perfil_B.filter(F.col("TIPO_CLIENTE") == "Menos Frequente")
        .select(cols_show).dropDuplicates(["nome_contratante"]).limit(5))

## Task C: Análise por Faixa de Capital
Comparativo demográfico:
* **Alto Capital:** Acima do 3º Quartil.
* **Baixo Capital:** Abaixo do 1º Quartil.

In [0]:
import pyspark.sql.functions as F

print("--- TASK C ---")

df_C_alto = df_prep.filter(F.col("ACIMA_Q3_CAPITAL") == True)
df_C_baixo = df_prep.filter(F.col("ABAIXO_Q1_CAPITAL") == True)

df_C_alto = df_C_alto.join(df_freq.select("nome_contratante", "qtd_acidentes"), "nome_contratante", "left").fillna(0, subset=["qtd_acidentes"])
df_C_baixo = df_C_baixo.join(df_freq.select("nome_contratante", "qtd_acidentes"), "nome_contratante", "left").fillna(0, subset=["qtd_acidentes"])

print("\n--- GRUPO ALTO CAPITAL (>Q3) ---")

print("1. Por Região:")
display(df_C_alto.groupBy("REGIAO").count().orderBy(F.col("count").desc()))

print("2. Por Sexo:")
display(df_C_alto.groupBy("SEXO").count())

print("3. Média de Acidentes (Ricos):")
df_media_ricos = df_C_alto.agg(F.avg("qtd_acidentes").alias("media_acidentes"))
display(df_media_ricos.withColumn("media_acidentes", F.round(F.col("media_acidentes"), 2)))


print("\n--- GRUPO BAIXO CAPITAL (<Q1) ---")

print("1. Por Região:")
display(df_C_baixo.groupBy("REGIAO").count().orderBy(F.col("count").desc()))

print("2. Média de Acidentes (Populares):")
df_media_pobres = df_C_baixo.agg(F.avg("qtd_acidentes").alias("media_acidentes"))
display(df_media_pobres.withColumn("media_acidentes", F.round(F.col("media_acidentes"), 2)))

## Task D: Eficiência da Carteira (Contratos por Acidente)
Métrica de rentabilidade/risco.
* **Valor Alto:** Região eficiente (muitos contratos para poucos acidentes).
* **Valor Baixo:** Região de risco (alta sinistralidade proporcional).

In [0]:
df_contratos = df_seguros_prep.groupBy("nome_contratante") \
    .agg(F.count("nome_contratante").alias("num_contratos"))

df_razao = df_contratos.join(
    df_freq.select("nome_contratante", "qtd_acidentes"), 
    on="nome_contratante", 
    how="left"
).fillna(0, subset=["qtd_acidentes"])

df_razao = df_razao.withColumn(
    "razao_contrato_acidente", 
    F.when(F.col("qtd_acidentes") > 0, F.col("num_contratos") / F.col("qtd_acidentes"))
    .otherwise(F.col("num_contratos"))
)

df_final_D = df_razao.join(
    df_seguros_prep.select("nome_contratante", "REGIAO", "SEXO").dropDuplicates(["nome_contratante"]),
    on="nome_contratante",
    how="inner"
)

print("📊 Eficiência Média por Região :")
display(
    df_final_D.groupBy("REGIAO")
    .agg(F.avg("razao_contrato_acidente").alias("media_razao"))
    .withColumn("media_razao", F.round(F.col("media_razao"), 5))
    .orderBy(F.col("media_razao").desc())
)